# `numpy`中的一元多项式(`polynomial.Polynomial`)对象, 以及支持的运算

In [1]:
#以别名形式导入numpy. 
import numpy as np; 
#导入与ndarray有关的数据类型. 
from numpy import int8, int16, int32, int64; 
from numpy import uint8, uint16, uint32, uint64; 
from numpy import float16, float32, float64; 
from numpy import complex64, complex128; 

In [2]:
from numpy import polynomial as nppn

## `numpy.polynomial.Polynomial`对象的构造
* 注意: `np.polynomial.Polynomial`只能用于定义标量的多项式, **不能**用于定义**矩阵多项式**. 

### 使用系数列表构造多项式
用法: 
```python
px = np.polynomial.Polynomial(iter_coef)
```
* `iter_coef` 包含多项式系数的迭代器
* 支持的迭代器类型: 
    * `list`, `tuple`, `numpy.ndarray`
    * 迭代器中不能内嵌迭代器; `numpy.ndarray`的`ndim`属性必须为1
* 迭代器内的所有元素应当为数值型对象, 否则将报错`ValueError`
    * Python内建的, 或`numpy`模块中定义的各类`int`, `float`, `complex`; 
    * `Decimal`, `Fraction`
    * 当迭代器为`list`或`tuple`时, 其中的元素类型不需要完全一致
* 系数按照自变量**次数从低到高**排列
    * 下标为`i`的元素表示`i`次项的系数
    * 除最高次项以外的其他任意项不存在时, 对应的系数为0, 必须列出

In [3]:
#定义一系列物理量
grav_Accl = 9.80665; #地表重力加速度(单位: m/s¹)

In [4]:
#计算自由落体运动的速度(单位: m/s)与时间(单位: s)的关系
velo_free_fall = nppn.Polynomial([0, grav_Accl]); 
velo_free_fall

Polynomial([0.     , 9.80665], domain=[-1,  1], window=[-1,  1])

### 使用零点列表构造多项式
用法: 
```python
px = np.polynomial.Polynomial.fromroots(iter_root)
```
* `iter_root` 包含多项式零点的迭代器
* 迭代器结构和元素类型要求与"[使用系数列表构造多项式](#使用系数列表构造多项式)"所用的迭代器相同
* 构造的多项式为$$\prod \limits_{i = 0}^{n - 1} (x - \mathrm{iter\_root}[i]),~\mathrm{其中} n = len(\mathrm{iter\_root})$$
    * 所构造多项式的最高次项系数为1
    * 多项式次数为构造期间使用的迭代器长度
* 迭代器中可以包含重复的元素, 同一元素重复$k$次, 表示所构造的多项式中对应的零点为$k$重零点
* 元素的顺序对所构造多项式的结果无影响

In [5]:
#构造具有零点的高阶多项式
if "random" not in globals(): import random; 
if "oper" not in globals(): import operator as oper; 
if "reduce" not in globals(): from functools import reduce; 
#随机生成8个零点, 均为-10≤x≤10的整数
poly_root = [random.randint(-10, 10) for i in range(8)]; 
print(poly_root); 
#利用零点构造16阶多项式
poly_High_Deg = nppn.Polynomial.fromroots(poly_root); 
#利用零点构造多项式的每个因式
poly_Factor = [nppn.Polynomial([-a, 1]) for a in poly_root]; 
#将所有因式连乘(不能使用np.prod)
poly_Expand = reduce(oper.mul, poly_Factor); 
print(poly_High_Deg == poly_Expand)
poly_High_Deg

[7, -4, -1, 5, -8, 1, 4, -6]
True


Polynomial([ 2.688e+04, -1.376e+03, -2.992e+04,  1.494e+03,  3.141e+03,
       -1.200e+02, -1.020e+02,  2.000e+00,  1.000e+00], domain=[-1.,  1.], window=[-1.,  1.])

## `numpy.polynomial.Polynomial`对象的调用和修改

### 计算多项式的值
* `Polynomial`对象可作为一元函数使用, 当`np.polynomial.Polynomial`被挂载至`poly`时, 
    ```python
    poly(x)
    ```
    返回多项式`poly`在自变量为`x`时的值
    * 求值过程通过秦九韶-Horner算法实现. 
    * 支持向量化运算. 如果`x`是`list`, `tuple`, `range`, `np.ndarray`等类型, 函数将对`x`中的每个元素求多项式的值, 并返回同型的`np.ndarray`对象. 

In [6]:
#计算自由落体的物体自释放后1s, 2s, 3s末的速度
print(velo_free_fall(range(1, 4)));

[ 9.80665 19.6133  29.41995]
